In [43]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [44]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline

In [45]:
#read all the words
words = open("C:/Users/Trader/Desktop/names.txt", 'r').read().splitlines()
words [:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [46]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [47]:
block_size = 3 
def build_dataset(words):
    # block size = context length aka how mnay tokesn used to predict. 
    X, Y = [],[]
    
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context) # current running context
            Y.append(ix) # prediction for the above context
            # print (''.join(itos[i] for i in context), '===>', itos[ix]) # print the context and the Y prediction
            context = context[1:] + [ix] # crop and append
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1]) #80%
Xdev,Ydev = build_dataset(words[n1:n2]) #10%

Xte, Yte = build_dataset(words[n2:]) # 10%

In [48]:
# manually comparing pytorch gradients to gradients that we calculate ourselves. 
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch. allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximater: {str(app):5s} | maxdiff: {maxdiff}')
    

In [49]:
n_embd = 10 #embedding matrix. 
n_hidden = 64 #neurons in the hidden layer

#generator to make it reproducable. 
g = torch.Generator().manual_seed(2147483647) 
# initialize all of the relevant tensors that we will need
C = torch.randn((vocab_size,n_embd),generator = g)
# layer 1
W1 = torch.randn((n_embd* block_size,n_hidden), generator = g) * (5/3.2) / (n_embd*block_size) **.05
b1 = torch.randn((n_hidden),generator = g) *.01
# layer 2
W2 = torch.randn((n_hidden,vocab_size),generator = g) *.01
b2 = torch.randn((vocab_size), generator = g) * 0.01 

#batch norm parameters 
bngain =  torch.randn((1,n_hidden))*.1 + 1
bnbias = torch.randn((1,n_hidden))*.1

bnmean_running = torch.zeros((1,n_hidden))
bnstd_running = torch.ones((1,n_hidden))

# model parameters
parameters = [C, W1, W2, b2, bngain, bnbias]

for p in parameters:
    p.requires_grad = True
sum (p.nelement() for p in parameters)

4073

In [50]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [51]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability (so that there are no extreme values these are all neg or 0)
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.2862, grad_fn=<NegBackward0>)

In [52]:
logprobs.shape

torch.Size([32, 27])

In [54]:
# backprop manually then checking to see if the pytorch derivatve and ours are the same. 
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n),Yb] = -1/n
dprobs = (1 / probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1,keepdim= True)
dcounts = counts_sum_inv * dprobs #broadcasting bc of different dimensions
dcounts_sum = -counts_sum**-2 * dcounts_sum_inv
# since this layer is just addition the derivative is passed through aka same as the 
#matrix before thus we just broadcast the prior derivatives into a matrix with the correct dimensions. 
dcounts += torch.ones_like(counts) * dcounts_sum 
dnorm_logits = counts * dcounts
dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1.0 - h**2) * dh
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
dbndiff += (2*bndiff) * dbndiff2
dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0)
dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
  for j in range(Xb.shape[1]):
    ix = Xb[k,j]
    dC[ix] += demb[k,j]


cmp('logprobs', dlogprobs ,logprobs)
cmp('probs', dprobs ,probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximater: True  | maxdiff: 0.0
probs           | exact: True  | approximater: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximater: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximater: True  | maxdiff: 0.0
counts          | exact: True  | approximater: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximater: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximater: True  | maxdiff: 0.0
logits          | exact: True  | approximater: True  | maxdiff: 0.0
h               | exact: True  | approximater: True  | maxdiff: 0.0
W2              | exact: True  | approximater: True  | maxdiff: 0.0
b2              | exact: True  | approximater: True  | maxdiff: 0.0
hpreact         | exact: True  | approximater: True  | maxdiff: 0.0
bngain          | exact: True  | approximater: True  | maxdiff: 0.0
bnbias          | exact: True  | approximater: True  | maxdiff: 0.0
bnraw           | exact: True  | approximater: T

TypeError: all() received an invalid combination of arguments - got (bool), but expected one of:
 * (Tensor input, *, Tensor out = None)
 * (Tensor input, tuple of ints dim = None, bool keepdim = False, *, Tensor out = None)
 * (Tensor input, int dim, bool keepdim = False, *, Tensor out = None)
